## Introducción

En estos ejercicios vamos a trabajar con el dataset alojado en la ruta del curso **/Dia-04/datasets/seguro_medico.csv** representa el conjunto de datos de clientes de una aseguradora de salud.

Las columnas del dataframe significan lo siguiente:

1. `age` : Edad del cliente
2. `sex` : Sexo del cliente
3. `bmi` : Índice de masa corporal del cliente (Ideal entre 18.5 y 24.9)
4. `children` : Número de hijos incluidos en el seguro de salud
5. `smoker` : El cliente es fumador o no 
6. `region` : Región donde vive el cliente en USA
7. `charges` : Gastos médicos individuales facturados por el seguro


Ejecuta la siguiente celda para el dataframe con el contenido del dataset.

In [48]:
import pandas as pd
df = pd.read_csv('./datasets/seguro_medico.csv')

## Ejercicio 1

**¿Cuál es el costo promedio del seguro de salud para fumadores y no fumadores?** 

Pista: Utiliza las funciones `groupby` y `describe`

In [49]:
promedio_fumador = df.groupby('smoker')['charges'].mean()

promedio_no_fumadores = promedio_fumador['no']
promedio_fumadores = promedio_fumador['yes']

print(f'El promedio de gastos médicos para NO fumadores es: {round(promedio_no_fumadores, 2)}')
print(f'El promedio de gastos médicos para fumadores es: {round(promedio_fumadores, 2)}')

El promedio de gastos médicos para NO fumadores es: 8434.27
El promedio de gastos médicos para fumadores es: 32050.23


## Ejercicio 2

**¿Cuál es la edad promedio de los asegurados por región?** 

Pista: Utiliza las funciones `groupby` y `describe`

In [50]:
edad_promedio_por_region = df.groupby('region')['age'].mean()

for region, edad_promedio in edad_promedio_por_region.items():
    print(f'Edad promedio en la región {region}: {round(edad_promedio, 2)}')

Edad promedio en la región northeast: 39.27
Edad promedio en la región northwest: 39.2
Edad promedio en la región southeast: 38.94
Edad promedio en la región southwest: 39.46


## Ejercicio 3

**¿Cuál es el grupo de personas que más paga en promedio si nos fijamos en la edad? ¿Y si nos fijamos en la edad y en el indice de masa corporal?** 

Pista: Utiliza las funciones `groupby`, `sort_values` y `iloc`

In [51]:
#Pregunta 1
promedio_costos_por_edad = df.groupby('age')['charges'].mean().sort_values(ascending=False)

edad_mayor_costo = promedio_costos_por_edad.index[0]
costo_mayor = promedio_costos_por_edad.iloc[0]

print(f'Grupo de edad que más paga: {edad_mayor_costo} años | costo: {round(costo_mayor, 2)}')

#Pregunta 2
promedio_costos_por_edad_imc = df.groupby(['age', 'bmi'])['charges'].mean().sort_values(ascending=False)

grupo_mayor = promedio_costos_por_edad_imc.index[0]
costo_mayor = promedio_costos_por_edad_imc.iloc[0]

print(f'Grupo de edad: {grupo_mayor[0]} años, IMC: {grupo_mayor[1]} | costo: {round(costo_mayor, 2)}')



Grupo de edad que más paga: 64 años | costo: 23275.53
Grupo de edad: 54 años, IMC: 47.41 | costo: 63770.43


## Ejercicio 4

**¿Cuál es el porcentaje de fumadores en cada región y género?** 

Pista: Utiliza las funciónes `groupby` y `size`

Documentación de `size`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.size.html

![descripción](../../imgs/division.png)

In [52]:
total_personas = df.groupby(['region', 'sex']).size()
total_fumadores = df[df['smoker'] == 'yes'].groupby(['region', 'sex']).size()

porcentaje_fumadores = (total_fumadores / total_personas) * 100

print('REGIÓN    | GÉNERO    | % DE FUMADORES')
for (region, sexo), porcentaje in porcentaje_fumadores.items():
    print(f'{region} |  {sexo}   |    {round(porcentaje, 2)}%')

REGIÓN    | GÉNERO    | % DE FUMADORES
northeast |  female   |    18.01%
northeast |  male   |    23.31%
northwest |  female   |    17.68%
northwest |  male   |    18.01%
southeast |  female   |    20.57%
southeast |  male   |    29.1%
southwest |  female   |    12.96%
southwest |  male   |    22.7%


## Ejercicio 5

**¿Cómo varía el costo promedio del seguro de salud para fumadores y no fumadores en función de la cantidad de hijos que tienen?** 

Pista: Utiliza las funciones `groupby` y `mean` y `unstack`


In [53]:
costo_promedio_por_hijos_fumador = df.groupby(['children', 'smoker'])['charges'].mean().unstack()

for numero_hijos in costo_promedio_por_hijos_fumador.index:
    print(f'\nPara {numero_hijos} hijos:')
    print(f'  No fumadores: {round(costo_promedio_por_hijos_fumador.loc[numero_hijos, "no"], 2)}')
    print(f'  Fumadores: {round(costo_promedio_por_hijos_fumador.loc[numero_hijos, "yes"], 2)}')


Para 0 hijos:
  No fumadores: 7611.79
  Fumadores: 31341.36

Para 1 hijos:
  No fumadores: 8303.11
  Fumadores: 31822.65

Para 2 hijos:
  No fumadores: 9493.09
  Fumadores: 33844.24

Para 3 hijos:
  No fumadores: 9614.52
  Fumadores: 32724.92

Para 4 hijos:
  No fumadores: 12121.34
  Fumadores: 26532.28

Para 5 hijos:
  No fumadores: 8183.85
  Fumadores: 19023.26


## Ejercicio 6

**¿Cuál es la diferencia absoluta y relativa (porcentaje) en el costo del seguro entre fumadores y no fumadores en cada grupo de edad?** 

Pista: Utiliza las funciones `groupby` y `apply`

In [54]:
promedio_costos_por_edad_fumador = df.groupby(['age', 'smoker'])['charges'].mean().unstack()

diferencia_absoluta = promedio_costos_por_edad_fumador['yes'] - promedio_costos_por_edad_fumador['no']
diferencia_relativa = (diferencia_absoluta / promedio_costos_por_edad_fumador['no']) * 100

print('EDAD | DIFERENCIA ABSOLUTA | DIFERENCIA RELATIVA (%)')
for edad in promedio_costos_por_edad_fumador.index:
    print(f' {edad}  |      {round(diferencia_absoluta[edad], 2)}       |        {round(diferencia_relativa[edad], 2)}%')

EDAD | DIFERENCIA ABSOLUTA | DIFERENCIA RELATIVA (%)
 18  |      22258.57       |        692.3%
 19  |      23464.52       |        663.46%
 20  |      20901.22       |        569.03%
 21  |      12837.07       |        336.62%
 22  |      34822.27       |        1365.03%
 23  |      25755.37       |        430.62%
 24  |      23616.07       |        422.66%
 25  |      24030.59       |        433.2%
 26  |      18361.55       |        440.69%
 27  |      19860.48       |        342.36%
 28  |      28493.65       |        473.61%
 29  |      18476.09       |        292.14%
 30  |      22357.4       |        424.51%
 31  |      30336.67       |        662.51%
 32  |      20534.72       |        389.56%
 33  |      23857.17       |        348.48%
 34  |      24279.51       |        349.63%
 35  |      20950.68       |        294.37%
 36  |      27363.07       |        485.39%
 37  |      27846.76       |        348.3%
 38  |      22589.39       |        358.81%
 39  |      16186.66      